# Clinical Trial Analytics — Data Engineering Notebook

**Protocol:** NV-2024-CT · Phase III RCT · Oncology  
**Author:** Namratha Vardhineni  
**Portfolio:** namratharao22.github.io/-Portfolio.Nv/  
**LinkedIn:** linkedin.com/in/namratha-vardhinedi-11079420a

---

This notebook demonstrates the full data preparation and analysis workflow for a simulated Phase III clinical trial.

It covers:
- Loading and inspecting CDISC-style ADaM datasets (ADSL, ADAE)
- Enrollment trend analysis by arm and site
- Adverse event summarization by SOC and CTCAE grade
- Dropout and protocol deviation metrics
- TFL output status tracking
- Export of analysis-ready summary datasets

The final datasets power the HTML dashboard (`Dashboard/index.html`).

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#8b949e',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
})
print('Setup complete.')

## 2. Load datasets

In [ ]:
adsl = pd.read_csv('../Data/ADSL.csv')
adae = pd.read_csv('../Data/ADAE.csv')
tfl  = pd.read_csv('../Data/TFL_Tracker.csv')

print(f'ADSL (Subject-level): {adsl.shape}')
print(f'ADAE (Adverse Events): {adae.shape}')
print(f'TFL Tracker: {tfl.shape}')

In [ ]:
adsl.head()

In [ ]:
adae.head()

## 3. Enrollment analysis

In [ ]:
# Total enrollment KPIs
total_enrolled = len(adsl)
target = 1320
pct_enrolled = round(total_enrolled / target * 100, 1)
active_sites = adsl['SITE'].nunique()
dropout_rate = round(len(adsl[adsl['STATUS']=='Discontinued']) / total_enrolled * 100, 1)

print(f'Total enrolled:  {total_enrolled} / {target} ({pct_enrolled}%)')
print(f'Active sites:    {active_sites}')
print(f'Dropout rate:    {dropout_rate}%')
print(f'Treatment arm:   {len(adsl[adsl["ARM"]=="Treatment"])}')
print(f'Placebo arm:     {len(adsl[adsl["ARM"]=="Placebo"])}')

In [ ]:
# Monthly enrollment by arm
month_arm = adsl.groupby(['ENROLL_MONTH','ARM']).size().unstack(fill_value=0).reset_index()
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun'}
month_arm['Month'] = month_arm['ENROLL_MONTH'].map(month_names)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(month_arm))
w = 0.35
ax.bar(x - w/2, month_arm.get('Treatment', 0), width=w, label='Treatment', color='#4f9cf9', zorder=3)
ax.bar(x + w/2, month_arm.get('Placebo', 0),   width=w, label='Placebo',   color='#2a3140', zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(month_arm['Month'])
ax.set_title('Monthly enrollment by arm', color='#e6edf3', pad=12)
ax.set_ylabel('Patients enrolled')
ax.legend()
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('../Images/enrollment_by_arm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: Images/enrollment_by_arm.png')

In [ ]:
# Enrollment by site
site_counts = adsl.groupby('SITE').size().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(site_counts.index, site_counts.values, color='#4f9cf9', zorder=3)
ax.set_title('Enrollment by site', color='#e6edf3', pad=12)
ax.set_xlabel('Patients enrolled')
ax.grid(axis='x', zorder=0)
for bar, val in zip(bars, site_counts.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../Images/enrollment_by_site.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Adverse event analysis

In [ ]:
# AE count by SOC
soc_counts = adae.groupby('SOC').size().sort_values(ascending=False)
print('AE counts by System Organ Class:')
print(soc_counts.to_string())

In [ ]:
# AE grade distribution
grade_dist = adae['CTCAE_GRADE'].value_counts().sort_index()
total_ae = len(adae)
grade_pct = (grade_dist / total_ae * 100).round(1)
print('Grade distribution:')
for g, pct in grade_pct.items():
    print(f'  Grade {g}: {pct}%')

In [ ]:
# AE severity table — mirrors clinical TFL output
ae_pivot = adae.pivot_table(
    index='SOC', columns='CTCAE_GRADE', values='AEID',
    aggfunc='count', fill_value=0
).reset_index()
ae_pivot.columns = ['SOC'] + [f'Grade {c}' for c in ae_pivot.columns[1:]]
ae_pivot['Total'] = ae_pivot[[c for c in ae_pivot.columns if 'Grade' in c]].sum(axis=1)
ae_pivot = ae_pivot.sort_values('Total', ascending=False)

# SAE flag
sae_map = adae[adae['SERIOUS']=='Y'].groupby('SOC').size()
ae_pivot['SAE Count'] = ae_pivot['SOC'].map(sae_map).fillna(0).astype(int)
ae_pivot['SAE Flag'] = ae_pivot['SAE Count'].apply(lambda x: 'Yes' if x > 0 else 'No')

print('Adverse Event Summary by SOC and Grade (CTCAE v5.0):')
print(ae_pivot.to_string(index=False))

In [ ]:
# Grade distribution donut chart
grade_colors = ['#56d364','#f0a732','#ff7b72','#f85149']
fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    grade_pct.values, labels=[f'Grade {g}' for g in grade_pct.index],
    colors=grade_colors, autopct='%1.1f%%',
    pctdistance=0.75, wedgeprops={'width':0.5, 'edgecolor':'#0d1117', 'linewidth':2}
)
for t in autotexts: t.set_color('#e6edf3'); t.set_fontsize(10)
ax.set_title('AE grade distribution — CTCAE v5.0', color='#e6edf3', pad=14)
plt.tight_layout()
plt.savefig('../Images/ae_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Dropout and subject disposition

In [ ]:
disposition = adsl['STATUS'].value_counts()
print('Subject disposition:')
for status, count in disposition.items():
    pct = round(count/total_enrolled*100, 1)
    print(f'  {status}: {count} ({pct}%)')

fig, ax = plt.subplots(figsize=(7, 4))
colors_d = ['#4f9cf9','#3fd4a0','#f0a732','#f85149']
ax.bar(disposition.index, disposition.values, color=colors_d[:len(disposition)], zorder=3)
ax.set_title('Subject disposition', color='#e6edf3', pad=12)
ax.set_ylabel('Count')
ax.grid(axis='y', zorder=0)
for i, (val) in enumerate(disposition.values):
    ax.text(i, val + 1, str(val), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../Images/subject_disposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. TFL output tracker

In [ ]:
tfl_summary = tfl.groupby(['TYPE','STATUS']).size().unstack(fill_value=0)
print('TFL Output Status:')
print(tfl_summary)

tfl_total = tfl.groupby('TYPE').size()
tfl_validated = tfl[tfl['STATUS']=='Validated'].groupby('TYPE').size()
tfl_pct = (tfl_validated / tfl_total * 100).round(1)
print('\nValidation completion:')
for t, pct in tfl_pct.items():
    print(f'  {t}: {pct}% validated')

## 7. Export analysis-ready summary datasets

In [ ]:
# Enrollment summary
enroll_summary = adsl.groupby(['ENROLL_MONTH','ARM']).size().reset_index(name='COUNT')
enroll_summary.to_csv('../Data/Enrollment_Summary.csv', index=False)

# AE summary
ae_pivot.to_csv('../Data/AE_Summary_by_SOC.csv', index=False)

# Site summary
site_summary = adsl.groupby('SITE').agg(
    Total=('USUBJID','count'),
    Treatment=('ARM', lambda x: (x=='Treatment').sum()),
    Placebo=('ARM', lambda x: (x=='Placebo').sum()),
    Completed=('STATUS', lambda x: (x=='Completed').sum()),
    Discontinued=('STATUS', lambda x: (x=='Discontinued').sum())
).reset_index()
site_summary.to_csv('../Data/Site_Summary.csv', index=False)

print('Exported:')
print('  Data/Enrollment_Summary.csv')
print('  Data/AE_Summary_by_SOC.csv')
print('  Data/Site_Summary.csv')
print('\nAll done. Open Dashboard/index.html to view the interactive dashboard.')

---

## Key findings

| Metric | Value |
|---|---|
| Total enrolled | 200 / 1,320 target (sample) |
| Active sites | 6 |
| Dropout rate | ~8.3% |
| Total AEs | 473 across 7 SOCs |
| Most common SOC | Gastrointestinal disorders (142 events) |
| Grade 3–4 AEs | ~21% of all events |
| TFLs validated | 145 / 172 (84%) |

**Note:** This dataset is simulated for portfolio demonstration purposes. Data follows CDISC ADaM conventions (ADSL, ADAE domain structure) and CTCAE v5.0 grading terminology.